In [1]:
!pip install datasets pandas numpy

In [2]:
import pandas as pd
import numpy as np
from datasets import load_dataset

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Libraries loaded successfully!")

Pandas version: 2.2.3
NumPy version: 2.1.3
Libraries loaded successfully!


In [4]:
dataset = load_dataset(
    "theguywithblacktie/hinglish-conversations",
    "small",
    split="train[:100]"
)

print(dataset)

data/small/train.parquet: reconstructing file:   0%|          |  0.00B / 13.1MB            

data/small/train.parquet: downloading bytes:           |  0.00B            

data/small/test.parquet: reconstructing file:   0%|          |  0.00B / 1.47MB            

data/small/test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/45000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset({
    features: ['input', 'output'],
    num_rows: 100
})


In [5]:
df = dataset.to_pandas()

print("\nColumns:")
print(df.columns.tolist())

print("\nShape:")
print(df.shape)

print("\nFirst 5 rows:")
display(df.head())


Columns:
['input', 'output']

Shape:
(100, 2)

First 5 rows:


,input,output
0,"[{'role': 'user', 'content': 'I sat in a lot o...","[{'role': 'assistant', 'content': 'Haan, it's ..."
1,"[{'role': 'user', 'content': 'arre,mujhe toh y...","[{'role': 'assistant', 'content': 'yeh toh per..."
2,"[{'role': 'user', 'content': 'Hey Simran, yaar...","[{'role': 'assistant', 'content': 'Aur tumhari..."
3,"[{'role': 'user', 'content': 'kaash mere paren...","[{'role': 'assistant', 'content': 'haan, meri ..."
4,"[{'role': 'user', 'content': 'Arre yaar, Sunit...","[{'role': 'assistant', 'content': 'Rohan, Beng..."


In [6]:
print("Data types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nNumber of rows:", len(df))

print("\nExample input:")
print(df["input"].iloc[0])

print("\nExample output:")
print(df["output"].iloc[0])

Data types:
input     object
output    object
dtype: object

Missing values:
input     0
output    0
dtype: int64

Number of rows: 100

Example input:
[{'role': 'user', 'content': 'I sat in a lot of traffic the other day as there was a wreck up ahead. This crazy person decided he was more important and zoomed by in the emergency lane'}
 {'role': 'assistant', 'content': "Arre yaar! That's so frustrating! People are so mad, right? Like, seriously? The emergency lane is for *actual* emergencies, not to cut lines. Tu chupke se unka video le leta, then post it online! 😂"}
 {'role': 'user', 'content': 'Ya, it was just unbelievable! I was so annoyed. I wanted to say something, but then thought better of it.  It felt so unsafe'}]

Example output:
[{'role': 'assistant', 'content': "Haan, it's better to not get involved bhai, especially when people are behaving like that. Safety first!  It's good tu didn't say anything. Just let them go, and keep your peace of mind.  Bohat stress leta hai aise l

In [7]:
import sys

input_size = sum(
    len(str(x).encode("utf-8"))
    for x in df["input"]
)

output_size = sum(
    len(str(x).encode("utf-8"))
    for x in df["output"]
)

total_size = input_size + output_size

print("Input size:", round(input_size / (1024 * 1024), 4), "MB")
print("Output size:", round(output_size / (1024 * 1024), 4), "MB")
print("Total size:", round(total_size / (1024 * 1024), 4), "MB")

Input size: 0.0323 MB
Output size: 0.0214 MB
Total size: 0.0537 MB


In [8]:
large_dataset = load_dataset(
    "theguywithblacktie/hinglish-conversations",
    "large",
    split="train"
)

print(large_dataset)

data/large/train.parquet: reconstructing file:   0%|          |  0.00B /  106MB            

data/large/train.parquet: downloading bytes:           |  0.00B            

data/large/test.parquet: reconstructing file:   0%|          |  0.00B / 11.6MB            

data/large/test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/364841 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/40538 [00:00<?, ? examples/s]

Dataset({
    features: ['input', 'output'],
    num_rows: 364841
})


In [9]:
df_large = large_dataset.to_pandas()

print("Total rows:", len(df_large))
print("Columns:", df_large.columns.tolist())

# Calculate the UTF-8 size of each conversation
df_large["input_bytes"] = df_large["input"].apply(
    lambda x: len(str(x).encode("utf-8"))
)

df_large["output_bytes"] = df_large["output"].apply(
    lambda x: len(str(x).encode("utf-8"))
)

df_large["total_bytes"] = (
    df_large["input_bytes"] +
    df_large["output_bytes"]
)

print("\nTotal dataset size:")
print(
    round(df_large["total_bytes"].sum() / (1024 * 1024), 2),
    "MB"
)

print("\nAverage conversation size:")
print(
    round(df_large["total_bytes"].mean() / 1024, 2),
    "KB"
)

Total rows: 364841
Columns: ['input', 'output']

Total dataset size:
187.49 MB

Average conversation size:
0.53 KB


In [10]:
TARGET_MB = 50
TARGET_BYTES = TARGET_MB * 1024 * 1024

# Use NumPy to calculate cumulative data size
cumulative_bytes = np.cumsum(df_large["total_bytes"].to_numpy())

# Find how many rows fit within approximately 50 MB
num_rows_50mb = np.searchsorted(
    cumulative_bytes,
    TARGET_BYTES,
    side="right"
)

df_50mb = df_large.iloc[:num_rows_50mb].copy()

actual_bytes = df_50mb["total_bytes"].sum()
actual_mb = actual_bytes / (1024 * 1024)

print("Rows selected:", len(df_50mb))
print("Target size:", TARGET_MB, "MB")
print("Actual size:", round(actual_mb, 2), "MB")

Rows selected: 97580
Target size: 50 MB
Actual size: 50.0 MB


In [11]:
print("Dataset shape:", df_50mb.shape)

print("\nColumns:")
print(df_50mb.columns.tolist())

print("\nMissing values:")
print(df_50mb[["input", "output"]].isnull().sum())

print("\nConversation size statistics:")
print(
    df_50mb["total_bytes"].describe()
)

print("\nTotal size:")
print(
    round(df_50mb["total_bytes"].sum() / (1024 * 1024), 2),
    "MB"
)

Dataset shape: (97580, 5)

Columns:
['input', 'output', 'input_bytes', 'output_bytes', 'total_bytes']

Missing values:
input     0
output    0
dtype: int64

Conversation size statistics:
count    97580.000000
mean       537.290234
std        504.800675
min        108.000000
25%        313.000000
50%        351.000000
75%        572.000000
max      13355.000000
Name: total_bytes, dtype: float64

Total size:
50.0 MB


In [12]:
print("\nFirst conversation:")
print(df_50mb[["input", "output"]].iloc[0])

print("\nLast conversation:")
print(df_50mb[["input", "output"]].iloc[-1])


First conversation:
input     [{'role': 'user', 'content': 'My mother-in-law...
output    [{'role': 'assistant', 'content': 'Arre yaar, ...
Name: 0, dtype: object

Last conversation:
input     [{'role': 'user', 'content': 'bhaiya,aapki wif...
output    [{'role': 'assistant', 'content': 'aree, apne ...
Name: 97579, dtype: object


In [13]:
def extract_text(messages):
    if not isinstance(messages, (list, np.ndarray)):
        return ""

    texts = []

    for message in messages:
        if isinstance(message, dict) and "content" in message:
            texts.append(str(message["content"]))

    return " ".join(texts)


df_50mb["input_text"] = df_50mb["input"].apply(extract_text)
df_50mb["output_text"] = df_50mb["output"].apply(extract_text)

print("Input example:")
print(df_50mb["input_text"].iloc[0])

print("\nOutput example:")
print(df_50mb["output_text"].iloc[0])

Input example:
My mother-in-law offered to babysit our kids about a month ago. When the day finally came, she acted like we had never asked her

Output example:
Arre yaar, woh toh bohot common hai! Mummy-ji log aise hi karte hain. Pehle bolti hai, phir bolti hai 'kya?' Matlab, tension mat le. Itni choti baat hai


In [14]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    # Remove unnecessary control characters
    text = "".join(
        char for char in text
        if char.isprintable()
    )

    return text.strip()


df_50mb["input_clean"] = df_50mb["input_text"].apply(clean_text)
df_50mb["output_clean"] = df_50mb["output_text"].apply(clean_text)

print("Original input:")
print(df_50mb["input_text"].iloc[0])

print("\nCleaned input:")
print(df_50mb["input_clean"].iloc[0])

print("\nOriginal output:")
print(df_50mb["output_text"].iloc[0])

print("\nCleaned output:")
print(df_50mb["output_clean"].iloc[0])

Original input:
My mother-in-law offered to babysit our kids about a month ago. When the day finally came, she acted like we had never asked her

Cleaned input:
My mother-in-law offered to babysit our kids about a month ago. When the day finally came, she acted like we had never asked her

Original output:
Arre yaar, woh toh bohot common hai! Mummy-ji log aise hi karte hain. Pehle bolti hai, phir bolti hai 'kya?' Matlab, tension mat le. Itni choti baat hai

Cleaned output:
Arre yaar, woh toh bohot common hai! Mummy-ji log aise hi karte hain. Pehle bolti hai, phir bolti hai 'kya?' Matlab, tension mat le. Itni choti baat hai


In [15]:
empty_input = (df_50mb["input_clean"].str.len() == 0).sum()
empty_output = (df_50mb["output_clean"].str.len() == 0).sum()

print("Empty inputs:", empty_input)
print("Empty outputs:", empty_output)

print("\nTotal conversations:", len(df_50mb))

Empty inputs: 0
Empty outputs: 0

Total conversations: 97580


In [16]:
df_50mb["conversation_text"] = (
    df_50mb["input_clean"] + " ||| " + df_50mb["output_clean"]
)

duplicate_count = df_50mb["conversation_text"].duplicated().sum()

print("Total conversations:", len(df_50mb))
print("Duplicate conversations:", duplicate_count)
print("Unique conversations:", len(df_50mb) - duplicate_count)

Total conversations: 97580
Duplicate conversations: 196
Unique conversations: 97384


In [17]:
df_clean = df_50mb.drop_duplicates(
    subset=["conversation_text"]
).copy()

print("Before:", len(df_50mb))
print("After:", len(df_clean))
print("Removed:", len(df_50mb) - len(df_clean))

Before: 97580
After: 97384
Removed: 196


In [18]:
hinglish_words = [
    "hai", "hain", "ho", "haan", "nahi", "nahin",
    "mujhe", "mujhse", "tum", "aap", "apka",
    "mera", "meri", "mere", "tera", "teri",
    "kya", "kyun", "kaise", "kaisa",
    "bahut", "bohot", "yaar", "bhai",
    "acha", "accha", "achha", "arre",
    "matlab", "wala", "wali", "wale",
    "kar", "karo", "karna", "raha", "rahi",
    "tha", "thi", "the", "toh", "bhi",
    "se", "ko", "ke", "ki", "ka",
    "mein", "me", "par", "nahi"
]

pattern = r"\b(" + "|".join(hinglish_words) + r")\b"

df_clean["hinglish_matches"] = (
    df_clean["input_clean"].str.lower().str.count(pattern)
    +
    df_clean["output_clean"].str.lower().str.count(pattern)
)

print("Total conversations:", len(df_clean))

print(
    "Conversations with H-English indicators:",
    (df_clean["hinglish_matches"] > 0).sum()
)

print(
    "Conversations without indicators:",
    (df_clean["hinglish_matches"] == 0).sum()
)

print("\nAverage H-English indicators per conversation:",
      df_clean["hinglish_matches"].mean())

Total conversations: 97384
Conversations with H-English indicators: 97381
Conversations without indicators: 3

Average H-English indicators per conversation: 17.396071223198884


In [19]:
# Keep only the columns needed for the final dataset

final_df = df_clean[
    ["input_clean", "output_clean"]
].copy()

print("Final columns:")
print(final_df.columns.tolist())

print("\nFinal conversations:", len(final_df))

Final columns:
['input_clean', 'output_clean']

Final conversations: 97384


In [20]:
# Calculate final UTF-8 sizes

final_df["input_bytes"] = final_df["input_clean"].apply(
    lambda x: len(x.encode("utf-8"))
)

final_df["output_bytes"] = final_df["output_clean"].apply(
    lambda x: len(x.encode("utf-8"))
)

final_df["total_bytes"] = (
    final_df["input_bytes"] +
    final_df["output_bytes"]
)

final_size_mb = (
    final_df["total_bytes"].sum()
    / (1024 * 1024)
)

print("Final cleaned size:", round(final_size_mb, 2), "MB")

Final cleaned size: 40.3 MB


In [21]:
final_df["input_chars"] = final_df["input_clean"].str.len()
final_df["output_chars"] = final_df["output_clean"].str.len()

final_df["total_chars"] = (
    final_df["input_chars"] +
    final_df["output_chars"]
)

print("Total conversations:", len(final_df))

print("\nTotal characters:",
      final_df["total_chars"].sum())

print("\nAverage characters/conversation:",
      round(final_df["total_chars"].mean(), 2))

print("\nMinimum characters:",
      final_df["total_chars"].min())

print("\nMaximum characters:",
      final_df["total_chars"].max())

print("\nMedian characters:",
      final_df["total_chars"].median())

Total conversations: 97384

Total characters: 42190252

Average characters/conversation: 433.24

Minimum characters: 37

Maximum characters: 9979

Median characters: 280.0


In [22]:
final_dataset = final_df[
    ["input_clean", "output_clean"]
].rename(
    columns={
        "input_clean": "input",
        "output_clean": "output"
    }
).copy()

print("Final shape:", final_dataset.shape)
print("\nFinal columns:")
print(final_dataset.columns.tolist())

display(final_dataset.head())

Final shape: (97384, 2)

Final columns:
['input', 'output']


,input,output
0,My mother-in-law offered to babysit our kids a...,"Arre yaar, woh toh bohot common hai! Mummy-ji ..."
1,ek naya trend hai ki agar tum chaat masala ko ...,"wow, yeh toh interesting hai! main toh hamesha..."
2,"Hey Ankita! Mera favorite cafe hai ""Chai Pe Ch...","I'm looking forward to it, Aditya! Kal ka din ..."
3,"yaar,chai mein ginger aur cardamom se zyada ky...","bilkul nahi, woh toh chai ko ek unique touch d..."
4,I got to work the other day all excited to wor...,That's understandable. Losing that anticipated...


In [23]:
output_file = "hinglish_clean.jsonl"

final_dataset.to_json(
    output_file,
    orient="records",
    lines=True,
    force_ascii=False
)

print("File created successfully:", output_file)

File created successfully: hinglish_clean.jsonl


In [24]:
import os

file_size_mb = os.path.getsize(output_file) / (1024 * 1024)

print("File:", output_file)
print("Size:", round(file_size_mb, 2), "MB")
print("Records:", len(final_dataset))

File: hinglish_clean.jsonl
Size: 42.63 MB
Records: 97384


In [25]:
# Read the exported JSONL back into Pandas

verified_df = pd.read_json(
    "hinglish_clean.jsonl",
    lines=True
)

print("Verified rows:", len(verified_df))
print("Verified columns:", verified_df.columns.tolist())

print("\nMissing values:")
print(verified_df.isnull().sum())

print("\nFirst record:")
display(verified_df.head(1))

Verified rows: 97384
Verified columns: ['input', 'output']

Missing values:
input     0
output    0
dtype: int64

First record:


,input,output
0,My mother-in-law offered to babysit our kids a...,"Arre yaar, woh toh bohot common hai! Mummy-ji ..."


In [26]:
from google.colab import files

files.download("hinglish_clean.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>